# Suite INS — Installs: schemas defined by transactions

A module defines its schemas in a module-scope `install(txn)` (object identity 86 in [CAP-115: schemas defined by idempotent installs](../../../mbse-basis/docs/capabilities/CAP-115.md#decided);
[CAP-115: schemas defined by idempotent installs](../../../mbse-basis/docs/capabilities/CAP-115.md)): its transaction upserts them, each schema object a symbol
of its meta-schema, named from the transaction (`txn.Schemas.OfObject`, 78), found by its name, else created, and its
spec written where it does not hold. Run again, an install writes nothing; another adds to what it finds. A schema with
no name of its own is defined in a spec where it is linked, `txn.Schemas.OfContainer(cb)`, which creates nothing.

As in Python, with the language's differences (docs/EQUIVALENCE.md): builders, the store and a transaction are `Proxy`
objects typed loosely, so callbacks are annotated `(t: any)`; ints are `bigint`s.

In [ ]:
import { Builders as B, JSON as Json, Mutations, Paths, Plain, Schemas } from "@mbse/schemas/Framework";
import { AttributeError, ValueError } from "@mbse/schemas/Framework/Errors";
import { repr } from "@mbse/schemas/Framework/Repr";
import { assert, entries, equal, raises, same } from "./support.js";

/** crm.Person, who knows persons since a year; crm.Shape, a union; crm.Grid, a container. */
function crm(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("crm/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Person: S.OfObject, Knows: S.OfRelation, Shape: S.OfUnion, Grid: S.OfContainer })
    .spec([
      (_: any) => _.Knows.name("crm.Knows").links([(t: any) => t.name("knower"), (t: any) => t.name("known")]).properties({
        since: S.OfInt }),
      (_: any) => _.Person.name("crm.Person").ref(true).properties({ name: S.OfStr }).adjacencies({
        knows: (a: any) => a.relation([_.Knows]).me("knower") }).adjacencies([
        (a: any) => a.relation([_.Knows]).me("known")]),
      (_: any) => _.Shape.name("crm.Shape").branches([
        (t: any) => t.name("circle").schema([S.OfFloat]), (t: any) => t.name("label").schema([S.OfStr])]),
      (_: any) => _.Grid.name("crm.Grid").index([S.OfInt]).element([S.OfFloat]),
    ])]))]);
}

/** crm.Person's age, added to what `crm` defined, and crm.Note, its text a property given nothing. */
function aged(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("crm/2").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Person: S.OfObject, Note: S.OfObject })
    .spec([(_: any) => _.Person.name("crm.Person").properties({ age: S.OfInt }),
      (_: any) => _.Note.name("crm.Note").properties({ text: (t: any) => t })])]))]);
}

/** x.Item, whose tags are a list of text, a container with no name of its own, defined where it is linked. */
function tagged(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("tags/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Item: S.OfObject })
    .spec([(_: any) => _.Item.name("x.Item").ref(true).properties({
      tags: S.OfContainer((c: any) => c.index([S.OfInt]).element([S.OfStr])) })])]))]);
}

## INS-01 · A module's install defines its schemas, each kind a symbol of its meta-schema; run again, it writes nothing

In [ ]:
{
  const store: any = new B.Store();
  store.transaction(crm);
  const [Person, Knows] = [store.registered("crm.Person"), store.registered("crm.Knows")];
  assert(Person.ref && same([...Person.properties.keys()], ["name"]) && Person.properties.get("name").schema === store.Schemas.OfStr.as_schema());
  assert(same([...Knows.entry.links.keys()], ["knower", "known"]) && same([...Knows.entry.properties.keys()], ["since"]));
  assert(Person.adjacencies.get("knows").relation === Knows && Person.adjacencies.get("knows").me === "knower");
  assert(same(store.registered("crm.Shape").branches.map((b: any) => b.name), ["circle", "label"]));
  const grid = store.registered("crm.Grid");
  assert(grid.index === store.Schemas.OfInt.as_schema() && grid.element === store.Schemas.OfFloat.as_schema());
  const snapshot = Json.dumps(Plain.ToPlain(store).Snapshot() as never);
  store.transaction(crm); // run again: each found by its name, nothing written
  assert(Json.dumps(Plain.ToPlain(store).Snapshot() as never) === snapshot);
  const ada = store.transaction(() => store.crm.Person((c: any) => c.name("Ada").knows((a: any) => a.insert((e: any) =>
    e.known((b: any) => b.name("Bob")).since(2020n))))); // what it defined builds data
  assert(same(entries(Person, ada, "knows").map((e) => `${e.get("known").name}`), ["Bob"]));
}

## INS-02 · A schema with no name of its own, defined in a spec where it is linked: made with the link, as the link's builder makes one, found holding when run again; linked twice, refused

In [ ]:
{
  const store: any = new B.Store();
  const outside = store.transaction(() =>
    store.Schemas.OfContainer((c: any) => c.index([store.Schemas.OfInt]).element([store.Schemas.OfStr]))); // made
  store.transaction(tagged);
  const tags = store.registered("x.Item").properties.get("tags").schema;
  assert(tags !== outside && tags.name === null); // defined where it is linked, made with the link
  assert(tags.index === store.Schemas.OfInt.as_schema() && tags.element === store.Schemas.OfStr.as_schema());
  store.transaction(tagged); // run again: an equal one is linked there, so nothing is made
  assert(store.registered("x.Item").properties.get("tags").schema === tags);
  const twice = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("twice/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Item: S.OfObject })
      .spec([(_: any) => ((texts: any) => _.Item.name("x.Twice").properties({ a: texts, b: texts }))(
        S.OfContainer((c: any) => c.index([S.OfInt]).element([S.OfStr])))])]))]);
  };
  raises(ValueError, () => store.transaction(twice), "an object a spec defines is linked where it is defined"); // one place
  const required = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("required/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Rule: S.OfPattern })
      .spec([(_: any) => _.Rule.name("x.Rule").requires([(c: any) => c.description("holds")])])]))]);
  };
  store.transaction(required); // a pattern's constraint is made as its builder makes one: its result, Bool
  assert(store.registered("x.Rule").requires[0].result === store.Schemas.OfBool.as_schema());
}

## INS-03 · An install adds to what another defined; a transaction given to an install names its own members, and the store's schemas besides

In [ ]:
{
  const store: any = new B.Store();
  store.transaction(crm);
  store.transaction(aged);
  assert(same([...store.registered("crm.Person").properties.keys()], ["name", "age"]));
  assert(store.registered("crm.Note").properties.get("text").schema === null); // an entry given nothing defines what it links
  const paired = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("pair/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Pair: S.OfUnion })
      .spec([(_: any) => _.Pair.name("x.Pair").branches([(t: any) => t.name("n").schema([S.OfInt, S.OfStr])])])]))]);
  };
  raises(TypeError, () => store.transaction(paired), "link 'schema' links one schema, got 2");
  /** x.Pairs, its entries a set of `a`, given as a builder takes it; and a unit on every object schema. */
  const units = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("units/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Pairs: S.OfRelation, Object: S.OfObject })
      .spec([
        (_: any) => _.Pairs.name("x.Pairs").links([(t: any) => t.name("a"), (t: any) => t.name("b")]).uniques("a"),
        (_: any) => _.Object.name("Schemas.OfObject").properties({ unit: S.OfStr }),
      ])]))]);
  };
  /** x.Meters, its unit the one `units` added. */
  const meters = (txn: any): any => txn.uid("meters/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Meters: txn.Schemas.OfObject })
    .spec([(_: any) => _.Meters.name("x.Meters").unit("m").description((d: any) => d.as_value("metres"))])]))]); // given whole, or visited
  store.transaction(units);
  store.transaction(meters);
  const pairs = store.registered("x.Pairs").uniques;
  assert(pairs.length === 1 && same([...pairs[0]], ["a"])); // a spec takes a builder's shorthand
  assert(new B.OfObject.Builder(store.registered("x.Meters"), false, store).unit.get() === "m" && store.registered("x.Meters").description === "metres");
  /** crm.Named, a pattern over crm.Person, which `crm` installed, its spec recorded over it. */
  const named = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("named/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Named: S.OfPattern })
      .spec([(_: any) => _.Named.name("crm.Named").symbols([(t: any) => t.name("a").schema([txn.crm.Person])]).spec(
        (_: any) => _.a.name("Ada"))])]))]); // one callback, or a list of them
  };
  /** x.Over, a pattern over x.Thing, which the same install defines: not yet there when its spec is recorded. */
  const unready = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("unready/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Thing: S.OfObject, Over: S.OfPattern })
      .spec([(_: any) => _.Over.name("x.Over").symbols([(t: any) => t.name("a").schema([_.Thing])]).spec([(_: any) => _.a])])]))]);
  };
  /** x.Bare, a pattern whose symbol names no schema. */
  const untyped = (txn: any): any => txn.uid("untyped/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Bare: txn.Schemas.OfPattern })
    .spec([(_: any) => _.Bare.name("x.Bare").symbols([(t: any) => t.name("a")]).spec([(_: any) => _.a])])]))]);
  /** x.Inline, a pattern whose symbol's schema is defined where it is linked. */
  const inline = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("inline/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Inline: S.OfPattern })
      .spec([(_: any) => _.Inline.name("x.Inline").symbols([(t: any) => t.name("a").schema([S.OfObject((o: any) => o)])]).spec([
        (_: any) => _.a])])]))]);
  };
  store.transaction(named);
  store.transaction(named); // run again: the spec it gives holds
  assert(store.registered("crm.Named").spec.length === 1 && same([...store.registered("crm.Named").spec[0].keys()], ["a"]));
  for (const install of [unready, untyped, inline]) {
    raises(ValueError, () => store.transaction(install),
      "symbol 'a' of a pattern whose spec a spec records names no schema the store holds");
  }
  const seen: [boolean, string][] = [];
  const looking = (txn: any): any => {
    seen.push([txn.Schemas.OfInt.as_schema() === store.Schemas.OfInt.as_schema(), repr(txn)]);
    raises(AttributeError, () => txn._hidden);
    assert(txn.then === undefined); // JavaScript's protocol probes: awaiting it does not throw
    const looked = txn.uid("look/1"); // gives back the transaction: what follows names the store's schemas still
    assert(looked.uid.get() === "look/1"); // read, a member is the builder's own
    const S = looked.Schemas;
    return looked.mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Int: S.OfScalar, Look: S.OfModule, Seen: S.OfObject })
      .spec([
        (_: any) => _.Int.name("Schemas.OfInt"), // the store's own, found
        (_: any) => _.Look.name("look").schemas({ Seen: _.Seen }), // a symbol by its key, though no one kind fills the link
        (_: any) => _.Seen.name("look.Seen"),
      ])]))]);
  };
  store.transaction(looking);
  assert(seen.length === 1 && seen[0]![0] && seen[0]![1] === "<transaction>" && store.registered("look.Seen").name === "look.Seen");
}

## INS-04 · The framework installs itself: a store is made by reading the installs of Schemas, Paths and Mutations; each, run again as a transaction, writes nothing

In [ ]:
{
  const store: any = new B.Store(); // made by the bootstrap, reading each install through the builders
  assert(store.registered("Mutations.OfProperty.Set").properties.get("value").schema === store.registered("Mutations.Payload"));
  assert(store.registered("Schemas.OfObject").adjacencies.get("properties").relation === store.registered("Schemas.HasProperty"));
  const held = new Map((store.names() as string[]).map((name) => [name, store.registered(name)]));
  for (const module of [Schemas, Paths, Mutations]) {
    store.transaction(module.install); // each found by its name; one it would change is refused (the store's own)
  }
  assert(store.names().length === held.size && (store.names() as string[]).every((name) => store.registered(name) === held.get(name)));
}

## INS-05 · An install removes members of the store's own schemas, each entry by its name: a property, an adjacency, a branch; run again, it writes nothing; a schema the store was made holding refuses it

In [ ]:
{
  const store: any = new B.Store();
  store.transaction(crm);
  store.transaction(aged);
  /** crm.Person without its age and its knows, crm.Shape without its label: each entry removed by its name. */
  const trimmed = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("crm/3").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Person: S.OfObject, Shape: S.OfUnion })
      .spec([
        (_: any) => _.Person.name("crm.Person").properties((a: any) => a.remove((e: any) => e.name("age")))
          .adjacencies((a: any) => a.remove((e: any) => e.name("knows"))),
        (_: any) => _.Shape.name("crm.Shape").branches((a: any) => a.remove((e: any) => e.name("label"))),
      ])]))]);
  };
  store.transaction(trimmed);
  const Person = store.registered("crm.Person");
  assert(same([...Person.properties.keys()], ["name"]) && Person.adjacencies.size === 0 && Person.anonymous.length === 1);
  assert(same(store.registered("crm.Shape").branches.map((b: any) => b.name), ["circle"]) && Person.validate().length === 0);
  const snapshot = Json.dumps(Plain.ToPlain(store).Snapshot() as never);
  store.transaction(trimmed); // run again: what it removes is absent, so it holds, and nothing is written
  assert(Json.dumps(Plain.ToPlain(store).Snapshot() as never) === snapshot);
  /** A member of Schemas.OfObject, a schema the store was made holding, removed. */
  const unref = (name: string) => (txn: any): any => txn.uid("unref/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([
    (p: any) => p.symbols({ Object: txn.Schemas.OfObject })
      .spec([(_: any) => _.Object.name("Schemas.OfObject").properties((a: any) => a.remove((e: any) => e.name(name)))])]))]);
  raises(TypeError, () => store.transaction(unref("ref")),
    "an extension only adds members to 'Schemas.OfObject': it removes or replaces what 'properties' holds");
  store.transaction(unref("nope")); // absent: its spec holds, and nothing is written
  assert(store.registered("Schemas.OfObject").properties.has("ref"));
}

## INS-06 · A union's branch names are unique: an install adding a branch under a name the union has for another schema, or described otherwise, is refused; the same branch again writes nothing; a part alike

In [ ]:
{
  const store: any = new B.Store();
  store.transaction(crm);
  /** crm.Shape gains a branch `name` of the basic scalar `scalar`, described where `description` is given. */
  const coded = (uid: string, name: string, scalar: string, description: string | null = null) => (txn: any): any => {
    const S = txn.Schemas;
    const branch = (t: any) => {
      t = t.name(name).schema([S[scalar]]);
      return description === null ? t : t.description(description);
    };
    return txn.uid(uid).mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Shape: S.OfUnion })
      .spec([(_: any) => _.Shape.name("crm.Shape").branches([branch])])]))]);
  };
  store.transaction(coded("bytes/1", "binary", "OfBytes"));
  const snapshot = Json.dumps(Plain.ToPlain(store).Snapshot() as never);
  store.transaction(coded("bytes/1", "binary", "OfBytes")); // run again: the same branch, so nothing is written
  store.transaction(coded("bytes/2", "binary", "OfBytes")); // another install adding it as it is: nothing either
  assert(Json.dumps(Plain.ToPlain(store).Snapshot() as never) === snapshot);
  raises(TypeError, () => store.transaction(coded("ints/1", "binary", "OfInt")),
    "an extension only adds members to 'crm.Shape': it adds branch 'binary' for 'Schemas.OfInt', which it has for 'Schemas.OfBytes'");
  raises(TypeError, () => store.transaction(coded("bytes/3", "binary", "OfBytes", "raw")),
    "an extension only adds members to 'crm.Shape': it adds branch 'binary' for 'Schemas.OfBytes' again, described otherwise");
  const Shape = store.registered("crm.Shape");
  assert(equal(Shape.branches.map((b: any) => [b.name, b.schema.name, b.description]), [
    ["circle", "Schemas.OfFloat", null], ["label", "Schemas.OfStr", null], ["binary", "Schemas.OfBytes", null]]));
  assert(Shape.validate().length === 0 && Json.dumps(Plain.ToPlain(store).Snapshot() as never) === snapshot);
  // The owner and the name determine the entry: from the owner's end, branches and parts are keyed by their names.
  assert(same(store.Schemas.HasBranch.as_schema().key("owner"), ["name"]) && same(store.Schemas.HasPart.as_schema().key("owner"), ["name"]));
  const [Int, Float, Text] = [store.Schemas.OfInt, store.Schemas.OfFloat, store.Schemas.OfStr];
  const [stamp] = store.transaction(() => { // in an imperative scope alike: an equal branch elided, another of a name held refused
    store.Schemas.OfUnion(Shape, (u: any) => u.branches([(t: any) => t.name("label").schema([Text])]));
    raises(TypeError, () => store.Schemas.OfUnion(Shape, (u: any) => u.branches([(t: any) => t.name("label").schema([Int])])),
      "it adds branch 'label' for 'Schemas.OfInt', which it has for 'Schemas.OfStr'");
    const stamp = store.Schemas.OfIntersection((i: any) => i.name("crm.Stamp").parts([
      (t: any) => t.name("at").schema([Int]), (t: any) => t.name("by").schema([Text]), (t: any) => t.name("by").schema([Text])]));
    store.Schemas.OfIntersection(stamp, (i: any) => i.parts((a: any) => a.insert((e: any) => e.name("by").schema(Text))));
    assert(same(stamp.parts.map((p: any) => p.name), ["at", "by"])); // an equal part elided as it is added, in a list or inserted
    raises(TypeError, () => store.Schemas.OfIntersection(stamp, (i: any) => i.parts((a: any) => a.insert((e: any) => e.name("at").schema(Float)))),
      "an extension only adds members to 'crm.Stamp': it adds part 'at' for 'Schemas.OfFloat', which it has for 'Schemas.OfInt'");
    return [stamp];
  });
  assert(same(Shape.branches.map((b: any) => b.name), ["circle", "label", "binary"]) && stamp.validate().length === 0);
}